# Layer Identification for Supervised Training

This notebook evaluates different attention layers to find the best `target_layer` for supervised attention alignment training.

## Setup

1. **Install dependencies**: Run cell 1 to install `agave_chem` and `torch`.
2. **Configure paths**: Update `PRETRAINED_MODEL_PATH` and `TRAINING_DATA_FILE` in cell 1.
3. **Set layer range**: Adjust `LAYER_RANGE` to the layers you want to evaluate.
4. Run all cells to find the layer with the lowest attention alignment loss.

## Usage

After identifying the best layer, use it as `TARGET_LAYER` in `supervised_training.ipynb`.

In [ ]:
!pip install git+https://github.com/denovochem/agave_chem.git -U --force-reinstall --no-cache-dir
!python -m pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install pandas

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from agave_chem.mappers.neural.constants import smiles_token_to_id_dict
from agave_chem.mappers.neural.model import AlbertWithAttentionAlignment, SupervisedConfig
from agave_chem.mappers.neural.tokenizer import CustomTokenizer
from model_training_scripts.albert_mapper_supervised_training import (
    SupervisedAtomMappingDataset,
    build_attention_target_from_mapped_rxn_smiles,
    evaluate_supervised_attention_loss,
)
from model_training_scripts.albert_mapper_unuspervised_training import MLMConfig

import torch
from transformers import AlbertForMaskedLM

TRAIN_PCT = 0.95
LAYER_RANGE = [8, 9, 10, 11]
PRETRAINED_MODEL_PATH = "/workspace/saved_models/albert-04-01-2026/checkpoint-epoch-9"
TRAINING_DATA_FILE = "/workspace/data/mcs_expert_mapped_rxns.txt"

tokenizer = CustomTokenizer(smiles_token_to_id_dict)

In [ ]:
rxns = []
with open(TRAINING_DATA_FILE, 'r') as handle:
  for line in handle:
    rxns.append(line.strip())

rxns_train = rxns[:int(len(rxns)*TRAIN_PCT)]
rxns_val = rxns[int(len(rxns)*TRAIN_PCT):]

In [ ]:
mlm_config = MLMConfig()

val_dataset = SupervisedAtomMappingDataset(
    texts=rxns_val,
    tokenizer=tokenizer,
    mlm_config=mlm_config,
    protected_tokens={"^", "$", ".", ">>"},
    max_length=256,
    use_random_smiles=False
)

In [ ]:
base_model = AlbertForMaskedLM.from_pretrained(PRETRAINED_MODEL_PATH)

supervised_config = SupervisedConfig(
    target_layer=0,
    multitask=False,
)

model = AlbertWithAttentionAlignment(
    base_model=base_model,
    supervised_config=supervised_config,
    max_length=256,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
best_layer = 0
best_loss = 1e10
for layer_num in LAYER_RANGE:
    layer_loss = evaluate_supervised_attention_loss(
        model, val_dataset, device=device, target_layer=layer_num
    )
    print(f"Layer {layer_num}: loss={layer_loss:.6f}")
    if layer_loss < best_loss:
        best_loss = layer_loss
        best_layer = layer_num
print(f"\nBest: layer={best_layer}, loss={best_loss:.6f}")